# Nested to Wide and Back

There are three ways to get wide data from a nested structure.
1. boolean ( True/False ) with the categories as columns
1. counts with the categories as columns
1. categories with counts as columns


From https://stackoverflow.com/a/69604744

In [ ]:
import pandas as pd
import numpy as np


Create a data frame with two columns, the seconds of which has nested lists, sometimes with duplicated entries.

Notice that "apple" appears twice in the second column of the first row.

In [ ]:
data = pd.DataFrame()
data['id'] = ["ab3e3", "psdds2", "pas13", "ccdf2", "dsda1"]
data['fruit'] = ["apple, orange, apple", "others", "dragon fruit, orange", "watermelon", "others"]

df = pd.DataFrame(data)
df


In [ ]:
print(df.to_markdown(index=False))

## Nested to Wide

### Boolean: Using .get_dummies() with a delimeter

Expand column categories into individual columns with binary values.

In [ ]:
df['fruit'].str.get_dummies(', ')


Join original data frame with get_dummies.

In [ ]:
df.join(df['fruit'].str.get_dummies(', '))


Join then drop "fruit".

In [ ]:
df_wide1 = (
  df
  .join(df['fruit'].str.get_dummies(', '))
  .drop(columns= ['fruit'])
)
df_wide1


Drop "fruit" then join.


In [ ]:
(
df
  .drop(columns=['fruit'])
  .join(df['fruit'].str.get_dummies(', '))
)

# Using .explode()

Explode a data series.

In [ ]:
df_explode = df.assign(fruit=df["fruit"].str.split(", ")).explode("fruit")
df_explode


In [ ]:
ct = df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").drop(columns=["id"])
ct

In [ ]:
ct["fruit"].unique()

In [ ]:
ct.value_counts()

In [ ]:
df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").value_counts()

In [ ]:
df_counts = df_explode.value_counts().reset_index()
df_counts


In [ ]:
np.repeat(df_counts.index, df_counts['count'])

In [ ]:
df_original = df_counts.reindex(df_counts.index.repeat(df_counts['count']))

In [ ]:
df_counts['count'], df_counts.index.repeat(df_counts['count'])

In [ ]:
df_counts.iloc[df_counts.index.repeat(df_counts['count'])]

In [ ]:
(
df_counts
  .reindex(df_counts.index.repeat(df_counts['count']))
  .drop(columns='count')
  .reset_index(drop=True)
)

In [ ]:
df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").value_counts().reset_index()


# Using .crosstab()

In [ ]:
pd.crosstab(ct.index, ct["fruit"])


In [ ]:
pd.crosstab(ct.index, ct["fruit"]).rename_axis(None, axis=1)


In [ ]:
pd.crosstab(ct.index, ct["fruit"]).rename_axis(None, axis=1).rename_axis(None, axis=0)


In [ ]:
df.join(pd.crosstab(ct.index, ct["fruit"]).rename_axis(None, axis=1).rename_axis(None, axis=0)).drop( columns = ["fruit"])
